In [ ]:
# STEP 1 - Generate an External Knowledge base (python.pdf)

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("C://knowledge_base//Bhagavad-gita_As_It_Is english.pdf")
pages = loader.load()

print("Metadata: \n", pages[0].metadata)

In [16]:
# STEP 2 - Split the knowledge base into documents

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", ",", " "]
)
split_docs = splitter.split_documents(pages)

print(f"Total chunks: ", {len(split_docs)})

Total chunks:  {7403}


In [ ]:
# STEP 3 - Create embeddings and vector store using FAISS

from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError(".env file not found.")

load_dotenv(env_path)
apiKey = os.getenv("OLLAMA_API_KEY")

embedding_model = OllamaEmbeddings(model="text-embedding-ada-002", api_key=apiKey)

vectors = FAISS.from_documents(split_docs, embedding=embedding_model)
print(vectors)

In [ ]:
# STEP 3 - Create embeddings and vector store using Chroma db

from langchain_community.vectorstores import Chroma

persistent_directory = "c:/content/chroma_db"

vectors = Chroma.from_documents(documents=split_docs,
                                     embedding=embedding_model,
                                     persist_directory=persistent_directory)
print(type(vectors))
print(vectors)

In [ ]:
# STEP 4 - Integrate vector database with LLM using Retrieval chain mechanism

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

custom_prompt_template = """
Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
--------------------------
{context}
Question: {question}
"""
CUSTOM_PROMPT = PromptTemplate(
    template=custom_prompt_template,
    input_variables=["context", "question"]
)
llm = ChatOllama(model="gpt-4o-mini")
qa_chain = RetrievalQA.from_chain_type(llm,
    chain_type="stuff",
    retriever=vectors.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": CUSTOM_PROMPT}
)

In [ ]:
# STEP 5 - Build CLI based interface

# Loop to ask multiple questions
while True:
    question = input("\nEnter the question (or type 'exit' to quit): ")

    if question.lower() in ['exit', 'quit']:
        print("Exiting... Have a great day!")
        break

    response = qa_chain({"query": question})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Show the answer
    print("\nAnswer:", answer)
    print(f"Source documents: {source_documents}")

In [ ]:
# STEP 6 - User interface

import gradio as gr

def chatbot_response(message, history):
    # Use the created RetrievalQA chain to get the answer
    response = qa_chain({"query": message})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Format the response to include the answer and source documents (optional)
    formatted_response = f"{answer}" # You can add source documents here if desired

    return formatted_response

# Create the Gradio interface
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="RAG Chatbot",
    description="Ask questions about Gen-AI and Langchain based on the provided text."
)

# Launch the interface
iface.launch(share=True)